# Lesson 4B — Cart Controller (Full-State Feedback)

*ESP2110 Inverted Pendulum Lab*

**Run in Google Colab:** open the notebook, run the **Setup** cell once, then run
cells top-to-bottom. No local files are required.

## Learning objectives
By the end of this notebook you can:
1. Explain why the Lesson 4A controller let the **cart drift**, and why feeding back *all four* states fixes it.
2. Check **controllability** and use **pole placement** to *choose* the closed-loop eigenvalues.
3. Regulate the cart to a **position setpoint** while keeping the pole upright, on linear and nonlinear plants.
4. Handle a **setpoint change** without destabilizing the pole.

### Parameters (same plant as Lesson 4A)
| Symbol | Meaning | Value |
| --- | --- | --- |
| `m_c` | Cart mass | 0.5 kg |
| `m_p` | Pole mass | 0.2 kg |
| `L` | Pole length | 0.3 m |
| `g` | Gravity | 9.81 m/s^2 |
| `dt` | Sample time | 0.01 s |

State vector: $x = [\,p,\ \dot p,\ \theta,\ \dot\theta\,]$, input force $f$, $\theta=0$ upright.

In [ ]:
# --- Setup (safe to re-run) ---
try:
    import numpy, scipy, matplotlib  # noqa: F401
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'matplotlib'], check=True)
print('Environment ready.')

---
## Lesson content

### Recap: why the cart drifted
In Lesson 4A we fed back only the **angle**: $f = K_p\theta + K_d\dot\theta$. That stabilized the
pole, but the closed-loop matrix had a **double eigenvalue at 0** for the cart — its position was
never penalized, so the cart drifted away while balancing.

The fix is **full-state feedback**: $f = -K\,(x - x_\text{ref})$ with a gain on *every* state
$[p,\dot p,\theta,\dot\theta]$. If the system is **controllable**, we can place all four
closed-loop eigenvalues wherever we want.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

m_c, m_p, L, g, dt = 0.5, 0.2, 0.3, 9.81, 0.01

def f_nonlin(s, f):
    p, v, th, om = s
    sin, cos = np.sin(th), np.cos(th)
    den = m_c + m_p * sin**2
    vdot  = (f + m_p * sin * (L * om**2 - g * cos)) / den
    omdot = (-f * cos - m_p * L * om**2 * sin * cos + (m_c + m_p) * g * sin) / (L * den)
    return np.array([v, vdot, om, omdot])

# Linearized model about the upright equilibrium (theta = 0)
A = np.array([
    [0, 1, 0,                       0],
    [0, 0, -m_p * g / m_c,          0],
    [0, 0, 0,                       1],
    [0, 0, (m_c + m_p) * g / (L * m_c), 0],
])
B = np.array([0, 1 / m_c, 0, -1 / (L * m_c)]).reshape(4, 1)
print('open-loop eigenvalues:', np.round(np.linalg.eigvals(A), 3))

## Part 1 - Is the system controllable?

Pole placement only works if the controllability matrix
$\mathcal{C} = [\,B\ AB\ A^2B\ A^3B\,]$ has full rank (4).

**Task:** build $\mathcal{C}$ and check its rank.

In [ ]:
# TODO: stack [B, A@B, A^2@B, A^3@B] horizontally and check the rank.
# Hints: np.linalg.matrix_power(A, i) @ B,  np.hstack(...),  np.linalg.matrix_rank(...)
C = ...
rank = ...
print('controllability matrix rank:', rank, '(need 4)')

The rank is **4**, so the cart-pole is **fully controllable** from the single force input — we are
free to place all four closed-loop eigenvalues.

## Part 2 - Pole placement: choose the closed-loop eigenvalues

We pick four **stable** target eigenvalues and ask `scipy.signal.place_poles` for the gain $K$ such
that $A - BK$ has exactly those eigenvalues. More negative poles = faster (but more aggressive).

**Task:** place the poles at $\{-2,-3,-4,-5\}$ and confirm the closed loop matches.

In [ ]:
# TODO: choose four negative target eigenvalues and compute K with place_poles.
target_poles = [..., ..., ..., ...]
K = signal.place_poles(A, B, target_poles).gain_matrix.ravel()
print('gain K =', np.round(K, 3))

cl_eig = np.linalg.eigvals(A - np.outer(B.ravel(), K))
print('closed-loop eigenvalues:', np.round(np.sort(cl_eig.real), 3))

The closed-loop eigenvalues come back as exactly **-5, -4, -3, -2** — placement worked. Note
every state now has a nonzero gain (including the cart position $p$), which is what lets us hold the
cart in place. Contrast with 4A, where the cart gains were 0 and it drifted.

## Part 3 - Regulate the cart to a setpoint

The control law is $f = -K\,(x - x_\text{ref})$ with $x_\text{ref} = [p_\text{ref}, 0, 0, 0]$.
We drive the cart to $p_\text{ref} = 0.5$ m from a small initial tilt, on **both** plants.

In [ ]:
def simulate(plant, K, x0, p_ref=0.0, T=6.0):
    n = int(T / dt)
    x = np.array(x0, dtype=float)
    ref = np.array([p_ref, 0.0, 0.0, 0.0])
    X = np.zeros((n, 4)); F = np.zeros(n)
    for k in range(n):
        f = float(-K @ (x - ref))
        X[k], F[k] = x, f
        if plant == 'linear':
            x = x + dt * (A @ x + B.ravel() * f)
        else:
            x = x + dt * f_nonlin(x, f)
    return np.arange(n) * dt, X, F

In [ ]:
# TODO: simulate both plants to p_ref = 0.5 m from x0 = [0, 0, 0.1, 0] and plot
#       cart position, pole angle, and control force (linear vs nonlinear).
x0 = [0.0, 0.0, 0.1, 0.0]
tl, Xl, Fl = simulate('linear',    K, x0, p_ref=0.5)
tn, Xn, Fn = simulate('nonlinear', K, x0, p_ref=0.5)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].axhline(0.5, color='gray', ls=':', label='p_ref')
ax[0].plot(tl, Xl[:, 0], label='linear'); ax[0].plot(tn, Xn[:, 0], '--', label='nonlinear')
ax[0].set(title='Cart position', xlabel='time (s)', ylabel='p (m)'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(tl, np.rad2deg(Xl[:, 2]), label='linear')
ax[1].plot(tn, np.rad2deg(Xn[:, 2]), '--', label='nonlinear')
ax[1].set(title='Pole angle', xlabel='time (s)', ylabel='theta (deg)'); ax[1].legend(); ax[1].grid(True)
ax[2].plot(tl, Fl, label='linear'); ax[2].plot(tn, Fn, '--', label='nonlinear')
ax[2].set(title='Control force', xlabel='time (s)', ylabel='f (N)'); ax[2].legend(); ax[2].grid(True)
fig.suptitle('Full-state feedback to p_ref = 0.5 m'); fig.tight_layout(); plt.show()

**Expected output.** The cart slides over and **settles at 0.5 m** while the pole returns to upright;
the **linear and nonlinear** curves nearly overlap (small angles throughout). Unlike 4A, the cart no
longer drifts - the position gain holds it on target. Peak force is modest (~1 N).

## Part 4 - Setpoint change

**Task:** command the cart to step from 0 -> 0.5 m at $t=2$ s, then back to 0 at $t=4$ s, and check
the pole stays upright through both transitions.

In [ ]:
# TODO: build a time-varying reference (0 -> 0.5 at t=2s -> 0 at t=4s) and simulate
#       the nonlinear plant, plotting cart position vs reference and the pole angle.
...

The cart tracks each step within ~1 s; the pole **tips a few degrees** during each move (the cart
has to lean to accelerate) and returns to upright once the cart settles. A larger/faster setpoint
jump demands a bigger transient tilt - that is the cart-vs-pole tradeoff.

## Part 5 - Watch it balance and move (animation)

Animate the **nonlinear** plant doing the 0 -> 0.5 m -> 0 maneuver. The cart should glide to each
target with the pole staying near vertical.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

frames = Xn[::10]; refs = R[::10]   # reuse the setpoint-change run from Part 4
fig, ax = plt.subplots(figsize=(6, 3))
ax.set_ylim(-0.15, 0.55); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Cart-pole setpoint change (nonlinear)'); ax.set_xlabel('x (m)')
cart = plt.Rectangle((-0.12, 0.0), 0.24, 0.12, fc='steelblue', ec='k'); ax.add_patch(cart)
pole, = ax.plot([], [], lw=3, color='crimson')
bob,  = ax.plot([], [], 'o', color='crimson', ms=9)
target = ax.axvline(0.0, color='green', ls=':', lw=1)

def _update(i):
    p, th = frames[i, 0], frames[i, 2]
    cart.set_xy((p - 0.12, 0.0))
    tip_x, tip_y = p + L * np.sin(th), 0.12 + L * np.cos(th)
    pole.set_data([p, tip_x], [0.12, tip_y]); bob.set_data([tip_x], [tip_y])
    target.set_xdata([refs[i], refs[i]])
    ax.set_xlim(-0.8, 1.2)
    return cart, pole, bob, target

anim = animation.FuncAnimation(fig, _update, frames=len(frames), interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

---
## Checkpoints
- Controllability matrix has **rank 4**.
- Closed-loop eigenvalues match your **target poles** (e.g. -2,-3,-4,-5).
- Cart **settles at the setpoint** (no drift) and pole returns to upright; linear ~ nonlinear.
- Through a setpoint change the pole stays within a few degrees and recovers.

## Common pitfalls
- **Targeting unstable/zero poles.** Every target eigenvalue must have a **negative** real part.
- **Too-fast poles.** Pushing poles very far left (e.g. -50) gives huge gains and saturating force.
- **Forgetting the reference offset.** Use $f=-K(x-x_\text{ref})$, not $-Kx$, or the cart parks at 0.
- **Sign of `B`.** `B` here is a column vector; use `B.ravel()` when adding `B*f` to a state vector.